In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Data Mining Data Lake Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales (1).csv


In [4]:
import os

os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver", exist_ok=True)

In [5]:
import shutil

shutil.copy(
    "/content/sales.csv",
    "/content/data_lake/bronze/sales.csv"
)

'/content/data_lake/bronze/sales.csv'

In [6]:
bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

In [7]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
df.show(5, truncate=False)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

In [9]:
before_count = df.count()
print("Records before preprocessing:", before_count)

Records before preprocessing: 1000


In [10]:
print(df.columns)

['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']


In [11]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 10


In [12]:
df_clean = df.dropDuplicates()

In [13]:
null_counts = df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [14]:
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

df_clean = df.fillna(
    {c: "Unknown" for c in text_columns}
)

In [15]:
df_clean = df_clean.fillna({
    "quantity": 0,
    "unit_price": 0.0,
    "discount_percent": 0.0
})

In [16]:
df_clean = df_clean.dropna(
    subset=["order_id"]
)

In [17]:
from pyspark.sql.functions import trim

for c in text_columns:
    df_clean = df_clean.withColumn(
        c,
        trim(col(c))
    )

In [18]:
df_clean.select(
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
).show(10, truncate=False)

+---------------+---------------------+---------+----------------+----------+--------------+------------+
|customer_name  |product              |category |payment_method  |city      |state         |order_status|
+---------------+---------------------+---------+----------------+----------+--------------+------------+
|Keerthi Mishra |Saree                |Clothing |Credit Card     |Mumbai    |Maharashtra   |Delivered   |
|Arjun Singh    |T-Shirt              |Clothing |Debit Card      |Chennai   |Tamil Nadu    |Delivered   |
|Keerthi Rao    |Hoodie               |Clothing |UPI             |Pune      |Maharashtra   |Shipped     |
|Tanvi Sharma   |Running Shoes        |Sports   |Net Banking     |Lucknow   |Uttar Pradesh |Cancelled   |
|Aarav Kumar    |Coffee 500g          |Grocery  |Debit Card      |Vijayawada|Andhra Pradesh|Delivered   |
|Aditya Malhotra|Biography            |Books    |Credit Card     |Pune      |Maharashtra   |Shipped     |
|Siddharth Menon|Biography            |Books  

In [19]:
from pyspark.sql.functions import initcap, lower

for c in ["customer_name", "product", "category", "city", "state"]:
    df_clean = df_clean.withColumn(
        c,
        initcap(lower(trim(col(c))))
    )

In [20]:
df_clean = df_clean.withColumn(
    "payment_method",
    initcap(lower(trim(col("payment_method"))))
)

df_clean = df_clean.withColumn(
    "order_status",
    initcap(lower(trim(col("order_status"))))
)

In [21]:
df_clean.select("quantity").show(10)

+--------+
|quantity|
+--------+
|       2|
|       6|
|       6|
|       6|
|      10|
|       7|
|       4|
|       6|
|       2|
|       4|
+--------+
only showing top 10 rows


In [22]:
invalid_quantity = df_clean.filter(
    col("quantity") < 0
).count()

print("Invalid quantity records:", invalid_quantity)

Invalid quantity records: 6


In [23]:
df_clean.filter(
    col("quantity") < 0
).show()

+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100198|      C0014| Arjun Pillai|              Hoodie|       Clothing|      -2|      2160|              12|     Net Banking|Vijayawada|Andhra Pradesh|2025-09-03|     Shipped|
|  100458|      C0216|  Lakshmi Das|            Football|         Sports|      -2|       800|              12|      Debit Card|Vijayawada|Andhra Pradesh|2026-09-03|     Shipped|
|  100593|      C0094|    Sai Patel|         Coffee 500g|        Grocery|      -1|       890|               5|

In [24]:
df_clean = df_clean.filter(
    col("quantity") >= 0
)

In [25]:
invalid_price = df_clean.filter(
    col("unit_price") < 0
).count()

print("Invalid unit price records:", invalid_price)

Invalid unit price records: 4


In [26]:
df_clean.filter(
    col("unit_price") < 0
).show()

+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|   customer_name|     product|   category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100235|      C0349|     Tanvi Reddy|  Power Bank|Electronics|       7|      -500|               0|           Upi|Vijayawada|Andhra Pradesh|2026-03-02|   Delivered|
|  100284|      C0302|    Ananya Reddy| Study Table|  Furniture|       2|      -500|              30|    Debit Card|Vijayawada|Andhra Pradesh|2026-06-22|   Delivered|
|  100487|      C0232|Keerthi Kulkarni|Office Chair|  Furniture|       4|      -500|              25|    Debit Card|    Mumbai|   Maharashtra|2025-02-05|    Returned

In [27]:
df_clean = df_clean.filter(
    col("unit_price") >= 0
)

In [28]:
invalid_discount = df_clean.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).count()

print("Invalid discount records:", invalid_discount)

Invalid discount records: 8


In [29]:
df_clean.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show()

+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|  customer_name|         product|       category|quantity|unit_price|discount_percent|  payment_method|      city|     state|order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|  100034|      C0192|   Divya Sharma|        Yoga Mat|         Sports|       7|      1720|              -5|Cash On Delivery|Coimbatore|Tamil Nadu|2025-08-19|   Delivered|
|  100140|      C0075|Swathi Tripathi|  Cooking Oil 5l|        Grocery|       4|       790|              -5|     Credit Card| Hyderabad| Telangana|2025-10-01|     Shipped|
|  100217|      C0214|  Neha Tripathi|        Football|         Sports|       1|       650|              -5|             Upi|   Chennai|Tami

In [30]:
df_clean = df_clean.filter(
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
)

In [31]:
df_clean.select("order_date").show(30, truncate=False)

+----------+
|order_date|
+----------+
|2026-01-01|
|2025-08-03|
|2025-05-06|
|2026-07-31|
|2026-08-30|
|2026-05-09|
|2025-12-02|
|2025-09-25|
|2025-09-04|
|2026-08-24|
|2026-07-27|
|2026-09-18|
|2025-08-16|
|2025-12-20|
|2026-06-30|
|2025-04-08|
|2026-05-25|
|2026-08-17|
|2026-05-11|
|2025-12-06|
|2026-06-29|
|2026-09-08|
|2026-03-24|
|2025-09-09|
|2026-09-01|
|2026-02-22|
|2026-07-01|
|2026-06-26|
|2026-09-17|
|2025-11-10|
+----------+
only showing top 30 rows


In [32]:
df_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = false)



In [38]:
from pyspark.sql.functions import col, coalesce, to_date, try_to_timestamp, lit

df_clean = df_clean.withColumn(
    "order_date_parsed",
    coalesce(
        to_date(try_to_timestamp(col("order_date"), lit("dd/MM/yyyy"))),
        to_date(try_to_timestamp(col("order_date"), lit("yyyy-MM-dd"))),
        to_date(try_to_timestamp(col("order_date"), lit("MM/dd/yyyy"))),
        to_date(try_to_timestamp(col("order_date"), lit("dd-MM-yyyy"))),
        to_date(try_to_timestamp(col("order_date"), lit("MM-dd-yyyy")))
    )
)

In [39]:
df_clean.select(
    "order_date",
    "order_date_parsed"
).show(30, truncate=False)

+----------+-----------------+
|order_date|order_date_parsed|
+----------+-----------------+
|2026-01-01|2026-01-01       |
|2025-08-03|2025-08-03       |
|2025-05-06|2025-05-06       |
|2026-07-31|2026-07-31       |
|2026-08-30|2026-08-30       |
|2026-05-09|2026-05-09       |
|2025-12-02|2025-12-02       |
|2025-09-25|2025-09-25       |
|2025-09-04|2025-09-04       |
|2026-08-24|2026-08-24       |
|2026-07-27|2026-07-27       |
|2026-09-18|2026-09-18       |
|2025-08-16|2025-08-16       |
|2025-12-20|2025-12-20       |
|2026-06-30|2026-06-30       |
|2025-04-08|2025-04-08       |
|2026-05-25|2026-05-25       |
|2026-08-17|2026-08-17       |
|2026-05-11|2026-05-11       |
|2025-12-06|2025-12-06       |
|2026-06-29|2026-06-29       |
|2026-09-08|2026-09-08       |
|2026-03-24|2026-03-24       |
|2025-09-09|2025-09-09       |
|2026-09-01|2026-09-01       |
|2026-02-22|2026-02-22       |
|2026-07-01|2026-07-01       |
|2026-06-26|2026-06-26       |
|2026-09-17|2026-09-17       |
|2025-11

In [40]:
invalid_dates = df_clean.filter(
    col("order_date_parsed").isNull()
).count()

print("Invalid date records:", invalid_dates)

Invalid date records: 5


In [41]:
df_clean.filter(
    col("order_date_parsed").isNull()
).select(
    "order_id",
    "order_date"
).show(20, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
+--------+----------+



In [42]:
df_clean = df_clean.drop("order_date")

df_clean = df_clean.withColumnRenamed(
    "order_date_parsed",
    "order_date"
)

In [43]:
df_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_status: string (nullable = false)
 |-- order_date: date (nullable = true)



In [44]:
df_clean.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_status|order_date|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |Delivered   |2026-01-01|
|100002  |C0042      |Arjun Singh    |T-shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |Delivered   |2025-08-03|
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |Upi             |P

In [45]:
null_counts_after = df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

null_counts_after.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_status|order_date|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|           0|         5|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+



In [46]:
final_duplicate_count = df_clean.count() - df_clean.dropDuplicates().count()

print("Duplicate rows remaining:", final_duplicate_count)

Duplicate rows remaining: 10


In [47]:
before_count = df.count()

In [48]:
after_count = df_clean.count()

print("Records before preprocessing:", before_count)
print("Records after preprocessing:", after_count)
print("Records removed:", before_count - after_count)

Records before preprocessing: 1000
Records after preprocessing: 982
Records removed: 18


In [49]:
print(
    "Negative quantity records:",
    df_clean.filter(col("quantity") < 0).count()
)

Negative quantity records: 0


In [50]:
print(
    "Negative unit price records:",
    df_clean.filter(col("unit_price") < 0).count()
)

Negative unit price records: 0


In [51]:
print(
    "Invalid discount records:",
    df_clean.filter(
        (col("discount_percent") < 0) |
        (col("discount_percent") > 100)
    ).count()
)

Invalid discount records: 0


In [52]:
silver_path = "/content/data_lake/silver/preprocessed_sales"

In [53]:
df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

In [54]:
import os

print(os.listdir(silver_path))

['_SUCCESS', '._SUCCESS.crc', 'part-00000-d85781c7-9583-4397-8b76-00b9d9ed1669-c000.csv', '.part-00000-d85781c7-9583-4397-8b76-00b9d9ed1669-c000.csv.crc']


In [55]:
silver_df = spark.read.csv(
    silver_path,
    header=True,
    inferSchema=True
)

In [56]:
silver_df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_status|order_date|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |Delivered   |2026-01-01|
|100002  |C0042      |Arjun Singh    |T-shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |Delivered   |2025-08-03|
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |Upi             |P

In [57]:
silver_count = silver_df.count()

print("Silver row count:", silver_count)

Silver row count: 982


In [59]:
print("Original row count :", before_count)
print("Cleaned row count  :", after_count)
print("Silver row count   :", silver_count)

print("Silver count matches cleaned data:", silver_count == after_count)

Original row count : 1000
Cleaned row count  : 982
Silver row count   : 982
Silver count matches cleaned data: True


In [60]:
import glob

csv_files = glob.glob(
    "/content/data_lake/silver/preprocessed_sales/*.csv"
)

print(csv_files)

['/content/data_lake/silver/preprocessed_sales/part-00000-d85781c7-9583-4397-8b76-00b9d9ed1669-c000.csv']


In [61]:
import shutil

silver_csv_file = csv_files[0]

shutil.copy(
    silver_csv_file,
    "/content/sales_silver.csv"
)

print("sales_silver.csv created successfully.")



sales_silver.csv created successfully.


In [62]:
import os

print(
    "sales_silver.csv exists:",
    os.path.exists("/content/sales_silver.csv")
)

sales_silver.csv exists: True


In [63]:
from google.colab import files

files.download("/content/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Preprocessing Summary

The following preprocessing operations were performed:

1. Removed exact duplicate rows.
2. Handled missing values in appropriate columns.
3. Trimmed extra spaces from text fields.
4. Standardized obvious capitalization inconsistencies.
5. Validated quantity and removed clearly invalid negative values.
6. Validated unit price and removed clearly invalid negative values.
7. Validated discount percentage and removed values outside the 0–100 range.
8. Converted valid order dates into a consistent date format.
9. Identified clearly invalid dates and represented them as NULL while preserving useful records.
10. Saved the processed dataset in the Silver layer.
11. Read the Silver dataset back using Spark for verification.
